# **GRU**

GRU — это тип рекуррентной нейросети (RNN), придуманный как более простой аналог LSTM, который умеет помнить важную информацию во времени и забывать лишнее.

Используется для:

временных рядов (цены, сигналы, сенсоры),

текста (NLP),

zₜ = σ(W_z·[hₜ₋₁, xₜ])
rₜ = σ(W_r·[hₜ₋₁, xₜ])

h̃ₜ = tanh(W·[rₜ ⊙ hₜ₋₁, xₜ])
hₜ = (1 − zₜ) ⊙ hₜ₋₁ + zₜ ⊙ h̃ₜ


zₜ — update gate

rₜ — reset gate

h̃ₜ — кандидат скрытого состояния

In [ ]:
import sys, os
sys.path.append(os.path.abspath("../.."))

In [ ]:
import torch.nn as nn
import torch
import torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from preprocessing.preprocess import prep, append_results
from preprocessing.target import ttp_target
from metrics.Metrics import merged_metrics
from preprocessing.ForTorch import set_seed, DEVICE, SeqDataset, torch_predict, train_torch_classifier, train_tabnet_ttp
name = "AFKS"
df: pd.DataFrame = pd.read_csv(f"/Users/side/Desktop/Trading Chaos AI/df/clean_df/{name}.csv")

In [ ]:
class GRUCls(nn.Module):
    def init(self, n_features, hidden=64, n_classes=3, dropout=0.2):
        super().init()
        self.gru = nn.GRU(n_features, hidden, batch_first=True)
        self.drop = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden, n_classes)

    def forward(self, x):
        out, _ = self.gru(x)
        h = self.drop(out[:, -1, :])
        return self.fc(h)

def train_gru_ttp(df, train_size, test_size, step, seq_len=64):
    set_seed(42)
    splitter = prep(
        df=df,
        target_fn=ttp_target, target_name="ttp", target_col="TTP_class",
        horizons=[12,24,48],
        train_size=train_size, test_size=test_size, step=step,
        target_kwargs={"n_classes":3},
        scale_cols=[
            "Open","High","Low","Close",
            "Alligator_Jaw","Alligator_Teeth","Alligator_Lips",
            "AO","AddOn_Anchor_Level","AddOn_Size_Pct"
        ]
    )

    for X_train, X_test, y_train, y_test, scaler in splitter:
        tr_ds = SeqDataset(X_train, y_train, seq_len)
        te_ds = SeqDataset(X_test,  y_test,  seq_len)
        tr_loader = DataLoader(tr_ds, batch_size=256, shuffle=True)
        te_loader = DataLoader(te_ds, batch_size=512, shuffle=False)

        model = GRUCls(n_features=X_train.shape[1], hidden=64, n_classes=3)

        y_pred = train_torch_classifier(model, tr_loader, te_loader, epochs=15, lr=1e-3)

        y_test_seq = y_test[seq_len-1:]
        metrics = merged_metrics(y_test_seq, y_pred)

        append_results({
            "task_type":"classification",
            "model_name":"GRU",
            "model_family":"rnn",
            "model_params":{"hidden":64,"seq_len":seq_len},
            "target_name":"ttp","target_variant":"3class","horizons":"12_24_48",
            **metrics
        })